# Aurora — professeur : examinateur (Kaggle)

Cette machine **ne s'entraîne jamais**. Elle fait passer des examens à un vrai modèle de l'application :

1. `diagnose` : mesure le modèle de base et publie ses points faibles par module, compétence et sous-compétence ;
2. `verify` : compare, sur des questions jamais vues, le modèle de base et le candidat entraîné ailleurs, puis rend un verdict (`promote`, `reject` ou `inconclusive`) selon une règle fixée à l'avance.

Les corrigés sont déterministes (exécution de tests, de requêtes SQL, comparateur de preuves des missions, égalité JSON typée…). Aucune note ne vient d'une loss ou d'un modèle juge. Les questions d'examen ne quittent pas cette machine : seuls des scores agrégés et des empreintes sortent.

Activer **GPU** et **Internet** dans les réglages du notebook. Pour `verify`, ajouter en entrée la sortie du notebook entraîneur ou le jeu de données privé contenant l'adaptateur.

Élèves connus :
- `qwen2.5:7b` → `Qwen/Qwen2.5-7B-Instruct` (business, media, conversation)
- `qwen2.5-coder:7b` → `Qwen/Qwen2.5-Coder-7B-Instruct` (code, missions)
- `qwen3-coder:30b` → `Qwen/Qwen3-Coder-30B-A3B-Instruct` (code, missions, cowork)
- `qwen3-coder-next:q4_K_M` → `Qwen/Qwen3-Coder-Next` (code, missions)
- `orcarouter/Qwen3.8-27B-Uncensored` → `orcarouter/Qwen3.8-27B-Uncensored` (conversation, academy, cyber, cowork, business, media, engineering)
- `qwen3-vl:8b` → `Qwen/Qwen3-VL-8B-Instruct` (missions)
- `qwen3-vl:30b` → `Qwen/Qwen3-VL-30B-A3B-Instruct` (media)
- `deepseek-r1:32b` → `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` (cyber, academy, code)

Modules :
- `missions` : Missions, CLI et agent : actions JSON, reçus, pointeurs, comptes rendus fondés
- `code` : Code : fonctions Python, correction de bugs, SQL, traçage, autres langages
- `conversation` : Conversation : connaissances et raisonnement en français, lecture, traduction, consignes
- `academy` : Academy : sciences, mathématiques, médecine, culture générale
- `cyber` : Cyber : connaissances défensives, classification CWE, renseignement sur la menace
- `business` : Business : calcul financier, lecture de rapports, sentiment
- `cowork` : Cowork : extraction de documents, appels d'outils, JSON conforme à un schéma
- `media` : Image, Dessin, Vidéo, Animation : compréhension et traduction des demandes visuelles
- `engineering` : 3D et Simulation : volumes, masses, tolérances, cinématique, unités


In [ ]:
ROLE = "diagnose"            # "diagnose" ou "verify"
STUDENT = "qwen2.5:7b"       # étiquette Ollama d'un modèle réellement appelé par Aurora
MODULES = None               # None = tous ; sinon par exemple "business,media,conversation"
PROFILE = "standard"         # "standard" ou "full" ("smoke" = répétition technique seulement)
CYCLE = 0                    # numéro du cycle : chaque cycle tire un examen de vérification inédit
BUDGET_MINUTES = 240

AURORA_OPTIONS = {"role": ROLE, "student": STUDENT, "profile": PROFILE, "modules": MODULES, "cycle": CYCLE,
                  "budget_minutes": BUDGET_MINUTES, "overrides": {}}
print(AURORA_OPTIONS)


In [ ]:
# Aurora professor - generated, private. Sources are embedded and hash-checked; no credential is stored here.
import base64, hashlib, io, json, os, re, shutil, stat, subprocess, sys, time, zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path, PurePosixPath

def extract_bundle(raw, destination, expected_sha256, max_bytes=100 * 1024 * 1024):
    """Validate all paths and hashes before writing; preserve any different existing file."""
    if len(raw) > max_bytes or hashlib.sha256(raw).hexdigest() != expected_sha256:
        raise ValueError("Source bundle hash or size mismatch")
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        names = archive.namelist()
        if len(set(names)) != len(names) or sum(i.file_size for i in archive.infolist()) > max_bytes:
            raise ValueError("Invalid or oversized source bundle")
        manifest = json.loads(archive.read("source_manifest.json"))
        if not isinstance(manifest, dict) or set(names) != set(manifest) | {"source_manifest.json"}:
            raise ValueError("Source manifest does not match archive")
        pending = []
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (str(relative) != info.filename or relative.is_absolute() or ".." in relative.parts or "\\" in info.filename
                    or stat.S_ISLNK(info.external_attr >> 16)
                    or (info.filename != "source_manifest.json" and relative.suffix != ".py")):
                raise ValueError("Unsafe source bundle member")
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination):
                raise ValueError("Source bundle escapes destination")
            content = archive.read(info)
            if info.filename in manifest and hashlib.sha256(content).hexdigest() != manifest[info.filename]:
                raise ValueError("Source file hash mismatch")
            if target.exists() and target.read_bytes() != content:
                raise ValueError("Existing different source preserved")
            pending.append((target, content))
        for target, content in pending:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                with target.open("xb") as stream:
                    stream.write(content)
    return destination


OPTIONS = dict(AURORA_OPTIONS)
SOURCE_SHA256 = '2b134c8ee5a714d1a7ed478799c4a91e66e42f0c6a6e3b77448d99748356b700'
SOURCES = base64.b64decode('...')
BASE = Path('/kaggle/working')
WORK = BASE / "aurora_professor"
WORK.mkdir(parents=True, exist_ok=True)
# Sources live outside the output folder: only evidence is published as kernel output.
REPO = extract_bundle(SOURCES, Path('/tmp/aurora_sources') / SOURCE_SHA256[:16], SOURCE_SHA256)
print("AURORA_BOOT " + json.dumps({"sources": SOURCE_SHA256, "role": OPTIONS["role"]}), flush=True)


def numbers(text):
    return tuple(int(part) for part in re.findall(r"\d+", text)[:3])


missing = []
for name, minimum in {'transformers': '4.57.0', 'peft': '0.13.0', 'bitsandbytes': '0.43.0', 'accelerate': '0.34.0', 'datasets': '3.0.0'}.items():
    try:
        current = version(name)
    except PackageNotFoundError:
        current = None
    if current is None or numbers(current) < numbers(minimum):
        missing.append(name + ">=" + minimum)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True, timeout=900)

env = dict(os.environ, PYTHONPATH=str(REPO), AURORA_PROFESSOR_ALLOW_CODE_EXECUTION="1", HF_HOME='/tmp/aurora_hf',
           TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONUNBUFFERED="1")
try:                                        # optional, only for gated base models
    from kaggle_secrets import UserSecretsClient
    env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

argv = [sys.executable, "-m", "auto_rl.professor.run", OPTIONS["role"], "--student", OPTIONS["student"],
        "--profile", OPTIONS["profile"], "--cycle", str(OPTIONS["cycle"]), "--work", str(WORK),
        "--budget-minutes", str(OPTIONS["budget_minutes"])]
if OPTIONS.get("modules"):
    argv += ["--modules", OPTIONS["modules"]]
if OPTIONS.get("skills"):
    argv += ["--skills", OPTIONS["skills"]]
for key, value in (OPTIONS.get("overrides") or {}).items():
    argv += ["--" + key.replace("_", "-"), str(value)]
inputs = Path('/kaggle/input')
if OPTIONS["role"] == "train":
    found = sorted(inputs.rglob("diagnosis.json")) if inputs.is_dir() else []
    if len(found) != 1:
        raise SystemExit("Expected exactly one diagnosis.json from the examiner, found %d" % len(found))
    argv += ["--diagnosis", str(found[0])]
if OPTIONS["role"] == "verify":
    found = sorted(inputs.rglob("adapter_config.json")) if inputs.is_dir() else []
    if len(found) != 1:
        raise SystemExit("Expected exactly one adapter from the trainer, found %d" % len(found))
    adapter = WORK / "training" / "adapter"
    if adapter.exists():
        shutil.rmtree(adapter)
    shutil.copytree(found[0].parent, adapter)
    for receipt in (found[0].parent.parent / "train_receipt.json", found[0].parent / "train_receipt.json"):
        if receipt.is_file():                # kernel output layout, then flat dataset layout
            shutil.copy2(receipt, adapter.parent / "train_receipt.json")
            break
    argv += ["--adapter", str(adapter)]

started = time.time()
# Lines are relayed one by one so a notebook cell shows progress, not only a script log.
process = subprocess.Popen(argv, env=env, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace")
beat = time.time()
with (WORK / "run.log").open("a", encoding="utf-8") as log:
    for line in process.stdout:
        log.write(line)
        log.flush()
        if line.startswith("AURORA_") or "Error" in line or "Traceback" in line or line.startswith("  File "):
            print(line, end="", flush=True)
        elif time.time() - beat > 120:           # a quiet phase (downloads, model load) stays observable
            beat = time.time()
            print("AURORA_HEARTBEAT " + json.dumps({"elapsed_seconds": round(beat - started), "last": line.strip()[-160:]}), flush=True)
code = process.wait()
if code:
    print("".join((WORK / "run.log").read_text(encoding="utf-8", errors="replace").splitlines(True)[-60:]), flush=True)
files = {}
for path in sorted(WORK.rglob("*")):
    if path.is_file():
        sha = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1 << 22), b""):
                sha.update(block)
        files[path.relative_to(WORK).as_posix()] = sha.hexdigest()
disk = {p: shutil.disk_usage(p).free // 2 ** 30 for p in ("/tmp", str(BASE)) if os.path.isdir(p)}
manifest = {"kind": "aurora_professor_remote_run", "options": OPTIONS, "sources_sha256": SOURCE_SHA256, "exit_code": code,
            "elapsed_seconds": round(time.time() - started, 1), "free_disk_gb": disk, "files": files}
(BASE / "aurora_professor_manifest.json").write_text(json.dumps(manifest, indent=2))
print("AURORA_DONE " + json.dumps({k: manifest[k] for k in ("exit_code", "elapsed_seconds", "free_disk_gb")}), flush=True)
if code:
    raise SystemExit(code)


In [ ]:
import json
from pathlib import Path
work = Path("/kaggle/working/aurora_professor")
if AURORA_OPTIONS["role"] == "diagnose":
    report = json.loads((work / "diagnosis.json").read_text())
    print("Examen complet :", report["complete"], "| précision macro :", round(report["summary"]["overall"]["macro_accuracy"], 4))
    for row in report["weak_points"]:
        print(f"{row['accuracy']:.3f}  [{row['ci95'][0]:.2f}, {row['ci95'][1]:.2f}]  n={row['items']:4d}  {row['skill']}" + ("" if row["trainable"] else "  (mesuré, sans données d'entraînement directes)"))
else:
    verdict = json.loads((work / f"verdict_{AURORA_OPTIONS['cycle']}.json").read_text())
    print("VERDICT :", verdict["verdict"], "| installable :", verdict["promotable"])
    for reason in verdict["reasons"]:
        print(" -", reason)
    for skill, table in verdict["skills"].items():
        print(f"{table['base_accuracy']:.3f} -> {table['candidate_accuracy']:.3f}  ({table['gain']:+.3f}, p={table['mcnemar_p']:.3f}, n={table['items']})  {skill}")
